In [1]:
!pip install -U langchain langchain-core==1.0 langchain-community langchain-experimental langchain_google_genai

  Using cached langchain-1.4.9-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_experimental-0.4.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached langchain_google_genai-4.4.3-py3-none-any.whl.metadata (2.7 kB)
INFO: pip is looking at multiple versions of langchain to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain-1.4.8-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain-1.4.7-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain-1.4.6-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain-1.4.5-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain-1.4.4-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain-1.4.3-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain-1.4.2-py3-none-any.whl.metadata (6.2 kB)
INFO: pip is still looking at multiple versions of langchain to determine which version is c

### Langchain Programming including setup in Cell 1

In [2]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

os.environ["GEMINI_API_KEY"]=userdata.get('gemini_key')

chat = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")


Prompt Templates

In [3]:
#Prompts
from langchain_core.prompts import PromptTemplate

# Simple prompt with placeholders
prompt_template = PromptTemplate.from_template(
    "Tell me a {adjective} joke about {content}."
)

# Filling placeholders to create a prompt
filled_prompt = prompt_template.format(adjective="funny", content="robots")
print(filled_prompt)
response = chat.invoke(filled_prompt)
print(response.content)

Tell me a funny joke about robots.
[{'type': 'text', 'text': 'Why did the robot go on a diet?\n\nBecause it had too much **heavy metal** in its system!', 'extras': {'signature': 'EmAKXgFpFH0Tx+WLRfByMvMF6C5kq1ypH0nXg77rs6Lf0QIq4MOfiAOtX+hA6pJTxeHMS0vCxD/JW6Zh/Lv8sGBcVoCF0jQWyQ6MXVkqNnkK+UHcuzNTBFD3DDIwf4tYfWs='}}]


### Using Memory and User sessions - Useful especially for Chatbots

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
model=ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
model

ChatGoogleGenerativeAI(output_version=None, model='models/gemini-2.5-flash-lite', google_api_key=SecretStr('**********'), client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x7ca18c088050>, default_metadata=(), model_kwargs={})

In [5]:
model.invoke("Hi , My name is Alex and I am an AI Engineer")

AIMessage(content="Hi Alex, it's great to meet you! As an AI, I find the field of AI engineering fascinating. What are some of the most exciting projects you're working on, or what aspects of AI engineering do you find most rewarding?", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--bb6ae59d-ed70-4617-a8d9-0e238259addc-0', usage_metadata={'input_tokens': 13, 'output_tokens': 50, 'total_tokens': 63, 'input_token_details': {'cache_read': 0}})

In [6]:
model.invoke("What is my name")

* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.
Please retry in 7h7m5.232527597s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  description: "Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits."
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash-lite"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 25625
}
].
* Quota exceeded for metric: generativelanguage.googleapis.com/generat

AIMessage(content='I do not have access to your personal information, including your name. I am a large language model, trained by Google, and I do not have the ability to know who you are.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--9524c480-1481-484e-ad1a-06c0d99847bb-0', usage_metadata={'input_tokens': 5, 'output_tokens': 38, 'total_tokens': 43, 'input_token_details': {'cache_read': 0}})

In [7]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [8]:
config={"configurable":{"session_id":"chat1"}}

In [9]:
from langchain_core.messages import HumanMessage
from langchain_core.messages import AIMessage

response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Alex and I am an AI Engineer")],
    config=config
)

* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.
Please retry in 7h6m50.347530769s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  description: "Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits."
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash-lite"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 25610
}
].
* Quota exceeded for metric: generativelanguage.googleapis.com/genera

In [10]:
response.content

"Hi Alex! It's great to meet you. As an AI, I find it fascinating to interact with AI Engineers.\n\nWhat kind of AI work are you involved in? I'm always interested in learning about new developments and applications in the field."

In [12]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.
Please retry in 7h4m57.901025608s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  description: "Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits."
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash-lite"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 25497
}
].
* Quota exceeded for metric: generativelanguage.googleapis.com/genera

KeyboardInterrupt: 

In [ ]:
## change the config-->session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

In [ ]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config1
)
response.content

In [ ]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

**Using Prompt templates and LCEL with History**


*Understand and Practice on your own*

---



In [ ]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Answer all the question to the best of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [ ]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Amit")]})

In [ ]:
chain.invoke({"messages":[HumanMessage(content="What's my name?")]})

In [ ]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [ ]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Jack")],
    config=config
)

response

In [ ]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

In [ ]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [ ]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Jack")],"language":"French"})
response.content

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [ ]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [ ]:
config = {"configurable": {"session_id": "chat4"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Jack")],"language":"French"},
    config=config
)
repsonse.content

In [ ]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config,
)

response.content